# 프로젝트 1 - Weekend 1: API와 체인 기반 FAQ 시스템 (Easy)

| 항목 | 내용 |
|------|------|
| **프로젝트** | 주택청약 FAQ 챗봇 |
| **소요 시간** | 5시간 (10사이클 x 30분) |
| **핵심 기술** | Python, OpenAI API, LangChain LCEL, Gradio |

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/rmaomina/llm_service_modu6/blob/main/Colab%20Notebooks/w1_api_and_chain_project/p1_weekend1_api_and_chain_0314_%E1%84%80%E1%85%B5%E1%86%B7%E1%84%86%E1%85%B5%E1%86%AB%E1%84%8B%E1%85%A1.ipynb)

## 환경 설정

In [1]:
# 아래 실습은 1차 작성 후, claude code로 수정 - 실습을 따라가며 개념을 다시 익히는 방식으로 작업했습니다.
# 필요한 패키지 설치
# -q 옵션: 설치 로그를 최소화 (quiet mode)
!pip install -q openai langchain-openai python-dotenv gradio

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 87.7/87.7 kB 2.2 MB/s eta 0:00:00


In [2]:
import os
from google.colab import userdata
os.environ["OPENAI_API_KEY"] = userdata.get('OPENAI_API_KEY')

# ── 주요 라이브러리 임포트 ────────────────────────────────────────
# OpenAI: 원시 API 클라이언트
from openai import OpenAI
# ChatOpenAI: LangChain이 OpenAI를 감싼 래퍼
from langchain_openai import ChatOpenAI
# ChatPromptTemplate: 변수가 포함된 프롬프트 템플릿
from langchain_core.prompts import ChatPromptTemplate
# StrOutputParser: LLM 응답에서 순수 문자열만 추출
from langchain_core.output_parsers import StrOutputParser

# ── 클라이언트 초기화 ─────────────────────────────────────────────
# client: OpenAI 원시 API 사용 시
client = OpenAI()
# llm: LangChain LCEL 체인 사용 시 (temperature=0 → 항상 일관된 답변)
llm = ChatOpenAI(model="gpt-4o-mini", temperature=0)

print("✅ 환경 설정 완료")
print(f"API 키: {'✅ 설정됨' if os.environ.get('OPENAI_API_KEY') else '❌ 없음'}")

✅ 환경 설정 완료
API 키: ✅ 설정됨


## 📦 실습용 샘플 데이터

In [35]:
# ============================================================
# 주택청약 FAQ 샘플 데이터 (실습용)
# 각 항목 구조:
#   id       : FAQ 고유 번호 (검색 결과 추적용)
#   category : 청약통장/청약자격/특별공급/일반공급/당첨·계약/기타
#   question : 실제 FAQ 질문
#   answer   : FAQ 답변 (\n으로 단계 구분)
#   keywords : 키워드 매칭 검색에 사용할 핵심 단어들
#   difficulty: easy/medium (사이클 2 탐색 실습용)
# ============================================================
SAMPLE_FAQ_DATA = [
    {"id": "FAQ001", "category": "청약통장",
     "question": "주택청약종합저축이란 무엇인가요?",
     "answer": "주택청약종합저축은 국민주택과 민영주택 모두에 청약할 수 있는 만능 통장입니다.\n1) 매월 2만원~50만원 자유 납입\n2) 가입 후 일정 기간 경과 시 청약 자격 부여\n3) 2009년 5월 이후 모든 청약통장이 통합됨",
     "keywords": ["청약종합저축", "만능통장", "납입", "가입"], "difficulty": "easy"},
    {"id": "FAQ004", "category": "청약통장",
     "question": "청약통장 1순위 조건은 무엇인가요?",
     "answer": "1순위 조건은 주택 유형에 따라 다릅니다.\n1) 민영주택: 수도권 12개월, 비수도권 6개월 + 예치금\n2) 국민주택: 수도권 12개월(24회), 비수도권 6개월(12회)\n3) 투기과열지구: 2년, 24회 납입",
     "keywords": ["1순위", "가입기간", "예치금", "투기과열지구"], "difficulty": "medium"},
    {"id": "FAQ005", "category": "청약자격",
     "question": "주택 청약 신청 자격 조건은 무엇인가요?",
     "answer": "1) 만 19세 이상 (기혼자는 연령 제한 없음)\n2) 청약통장 가입 필수\n3) 국민주택: 무주택 세대구성원\n4) 민영주택: 세대주 또는 세대원 가능\n※ 투기과열지구는 세대주만 청약 가능",
     "keywords": ["청약자격", "만19세", "무주택", "세대주"], "difficulty": "easy"},
    {"id": "FAQ006", "category": "청약자격",
     "question": "무주택자 기준은 무엇인가요?",
     "answer": "본인과 세대원 모두 주택 미소유 시 무주택자입니다.\n예외: 60세 이상 직계존속 소유 주택, 20㎡ 이하 소형주택, 상속 후 3개월 내 처분 주택\n※ 분양권/입주권도 주택 수에 포함",
     "keywords": ["무주택", "세대원", "소형주택", "분양권"], "difficulty": "medium"},
    {"id": "FAQ009", "category": "특별공급",
     "question": "특별공급의 종류에는 어떤 것이 있나요?",
     "answer": "1) 기관추천 (국가유공자, 장애인 등)\n2) 다자녀가구 (3명 이상)\n3) 신혼부부 (혼인 7년 이내)\n4) 생애최초 (최초 주택 구입)\n5) 노부모부양 (만 65세 이상 부모)\n※ 2021년부터 신혼/생애최초 물량 확대",
     "keywords": ["특별공급", "기관추천", "다자녀", "신혼부부", "생애최초"], "difficulty": "medium"},
    {"id": "FAQ010", "category": "특별공급",
     "question": "신혼부부 특별공급 조건은 무엇인가요?",
     "answer": "1) 혼인기간 7년 이내 무주택 세대주\n2) 소득: 도시근로자 월평균소득 100~140%\n3) 전용면적 85㎡ 이하\n4) 혼인기간 짧을수록 + 자녀 많을수록 가점 높음\n5) 예비 신혼부부도 신청 가능",
     "keywords": ["신혼부부", "혼인기간", "소득기준", "가점"], "difficulty": "medium"},
    {"id": "FAQ013", "category": "일반공급",
     "question": "가점제와 추첨제의 차이는 무엇인가요?",
     "answer": "가점제: 무주택기간+부양가족+가입기간으로 점수화 (84점 만점)\n추첨제: 무작위 추첨\n1) 투기과열지구: 가점제 100%\n2) 청약과열지역: 가점 75% + 추첨 25%\n3) 기타: 가점 40% + 추첨 60%",
     "keywords": ["가점제", "추첨제", "84점", "투기과열지구"], "difficulty": "medium"},
    {"id": "FAQ017", "category": "당첨/계약",
     "question": "당첨자 발표는 어떻게 확인하나요?",
     "answer": "1) 청약홈(www.applyhome.co.kr) 접속\n2) 당첨자 조회 메뉴 클릭\n3) 문자 알림 서비스 신청 가능\n※ 당첨 후 서류 제출 기간과 계약 일정 반드시 확인",
     "keywords": ["당첨자발표", "청약홈", "SMS알림", "서류제출"], "difficulty": "easy"},
    {"id": "FAQ020", "category": "당첨/계약",
     "question": "재당첨 제한이란 무엇인가요?",
     "answer": "당첨 후 일정 기간 다른 주택 청약 불가:\n1) 투기과열지구: 10년\n2) 청약과열지역: 7년\n3) 수도권 공공주택: 5년\n※ 세대원 전원 적용 (배우자 당첨 시 본인도 제한)",
     "keywords": ["재당첨제한", "10년", "7년", "세대원"], "difficulty": "medium"},
    {"id": "FAQ023", "category": "기타",
     "question": "청약홈 사이트는 어떻게 이용하나요?",
     "answer": "청약홈(www.applyhome.co.kr) - 한국부동산원 운영\n1) 회원가입 후 공인인증서/간편인증 로그인\n2) 청약 신청, 당첨 확인, 가점 계산 가능\n3) 모바일 앱(청약홈)도 동일 서비스 제공",
     "keywords": ["청약홈", "공인인증서", "간편인증", "가점계산"], "difficulty": "easy"},
]

# 검색 성능 측정용 테스트 질의 5개
# expected_faq_id: 검색이 올바르게 동작했을 때 가장 먼저 나와야 할 FAQ
SAMPLE_TEST_QUERIES = [
    {"query": "청약통장 가입하려면 어떻게 해요?", "expected_category": "청약통장", "expected_faq_id": "FAQ001"},
    {"query": "1순위 되려면 뭐가 필요해요?", "expected_category": "청약통장", "expected_faq_id": "FAQ004"},
    {"query": "신혼부부 특공 자격이 궁금해요", "expected_category": "특별공급", "expected_faq_id": "FAQ010"},
    {"query": "가점이 높으면 유리한가요?", "expected_category": "일반공급", "expected_faq_id": "FAQ013"},
    {"query": "당첨되면 어떻게 확인해요?", "expected_category": "당첨/계약","expected_faq_id": "FAQ017"},
    {"query": "역대 당첨자 중 최고 가점은?", "expected_category": "당첨/계약","expected_faq_id": "FAQ013"},
    {"query": "청약이 당첨되면 로또겠죠?", "expected_category": "","expected_faq_id": ""},
]

print(f"📦 FAQ 데이터 로드 완료: {len(SAMPLE_FAQ_DATA)}개 QA, {len(SAMPLE_TEST_QUERIES)}개 테스트 질의")

📦 FAQ 데이터 로드 완료: 10개 QA, 7개 테스트 질의


---
## 사이클 1: 첫 API 호출

OpenAI API로 주택청약 관련 질문을 보내고 답변을 받아보세요. `system` 역할에 "주택청약 전문 상담원"을 설정하세요.

In [4]:
# ── 사이클 1: 첫 API 호출 ─────────────────────────────────────────
# OpenAI Chat Completions API의 기본 구조를 익힙니다.
#
# messages 배열의 role 종류:
#   system : AI의 역할/성격/규칙을 설정 (사용자에게는 보이지 않음)
#   user   : 사용자의 발화
#   assistant: AI의 이전 답변 (멀티턴 대화에서 사용)

response = client.chat.completions.create(
    model="gpt-4o-mini",
    messages=[
        {
            "role": "system",
            # system 메시지: 페르소나와 행동 지침 설정⭐️
            "content": "당신은 주택청약 전문 상담원입니다. 친절하고 정확하게 안내해 주세요."
        },
        {
            "role": "user",
            "content": "주택청약종합저축이란 무엇인가요?"
        }
    ],
    temperature=0   # 0: 항상 동일한 답변 / 1.0 이상: 창의적이지만 변동성 있음
)

# 응답 객체 구조: response.choices[0].message.content에 답변 텍스트가 있음
answer = response.choices[0].message.content
print(f"💬 답변:\n{answer}")

# 토큰 사용량 확인 (API 비용 = 토큰 수에 비례)
print(f"\n📊 토큰 사용량: 입력 {response.usage.prompt_tokens} / 출력 {response.usage.completion_tokens}")

💬 답변:
주택청약종합저축은 주택을 구입하고자 하는 사람들을 위해 마련된 저축 상품으로, 주택청약을 통해 일정 금액을 저축하고, 이를 바탕으로 주택 청약을 신청할 수 있는 제도입니다. 이 저축 상품은 주택을 구매할 때 필요한 자금을 마련하는 데 도움을 주며, 청약 가점제에 따라 청약 순위를 결정하는 데도 영향을 미칩니다.

주택청약종합저축의 주요 특징은 다음과 같습니다:

1. **가입 대상**: 만 19세 이상의 대한민국 국민이면 누구나 가입할 수 있습니다.
2. **저축 금액**: 매월 일정 금액을 저축해야 하며, 최소 금액은 10,000원입니다. 최대 한도는 월 50만 원입니다.
3. **이자**: 저축한 금액에 대해 이자가 발생하며, 이자는 세제 혜택이 있습니다.
4. **청약 가점**: 저축 기간과 저축 금액에 따라 청약 가점이 부여되어, 주택 청약 시 우선 순위에 영향을 미칩니다.
5. **주택 구매**: 청약에 당첨되면, 해당 주택을 구매할 수 있는 기회를 얻게 됩니다.

주택청약종합저축은 주택 구매를 계획하는 분들에게 매우 유용한 제도이므로, 관심이 있으시다면 자세한 내용을 확인하고 가입을 고려해 보시는 것이 좋습니다. 추가적인 질문이 있으시면 언제든지 문의해 주세요!

📊 토큰 사용량: 입력 46 / 출력 342


---
## 사이클 2: FAQ 데이터 탐색

`SAMPLE_FAQ_DATA`에서 카테고리별 FAQ 개수를 세고, `difficulty`가 `"easy"`인 항목만 필터링해서 출력하세요.

In [5]:
# ── 사이클 2: FAQ 데이터 탐색 ─────────────────────────────────────
# 본격적인 검색 함수 작성 전, 데이터 구조를 파악합니다.
# Counter: 리스트에서 각 항목의 빈도를 딕셔너리 형태로 반환하는 유틸

from collections import Counter

# 카테고리별 FAQ 개수 집계
# ⭐️제너레이터 표현식: faq["category"] for faq in SAMPLE_FAQ_DATA
# ⭐️→ 리스트 전체를 메모리에 올리지 않고 순회하며 카운트
category_counts = Counter(faq["category"] for faq in SAMPLE_FAQ_DATA)
print("📊 카테고리별 FAQ 개수:")
for category, count in category_counts.items():
    print(f"  {category}: {count}개")

# 난이도 분포 확인
difficulty_counts = Counter(faq["difficulty"] for faq in SAMPLE_FAQ_DATA)
print(f"\n📈 난이도 분포: {dict(difficulty_counts)}")

# 리스트 컴프리헨션으로 easy 항목만 필터링
# [표현식 for 변수 in 리스트 if 조건] 패턴
easy_faqs = [faq for faq in SAMPLE_FAQ_DATA if faq["difficulty"] == "easy"]
print(f"\n✅ 난이도 'easy' FAQ ({len(easy_faqs)}개):")
for faq in easy_faqs:
    print(f"  [{faq['id']}] {faq['category']} - {faq['question']}")

📊 카테고리별 FAQ 개수:
  청약통장: 2개
  청약자격: 2개
  특별공급: 2개
  일반공급: 1개
  당첨/계약: 2개
  기타: 1개

📈 난이도 분포: {'easy': 4, 'medium': 6}

✅ 난이도 'easy' FAQ (4개):
  [FAQ001] 청약통장 - 주택청약종합저축이란 무엇인가요?
  [FAQ005] 청약자격 - 주택 청약 신청 자격 조건은 무엇인가요?
  [FAQ017] 당첨/계약 - 당첨자 발표는 어떻게 확인하나요?
  [FAQ023] 기타 - 청약홈 사이트는 어떻게 이용하나요?


---
## 사이클 3: FAQ 검색 함수

질문 문자열을 받아 키워드 매칭으로 관련 FAQ를 찾는 `search_faq(query, faq_data, top_k=3)` 함수를 만들고, `SAMPLE_TEST_QUERIES`로 테스트하세요.

In [37]:
# ── 사이클 3: FAQ 검색 함수 ───────────────────────────────────────
# 키워드 기반 검색(BM25/TF-IDF의 간소화 버전)을 직접 구현합니다.
# 2주차에서는 이 함수를 벡터 임베딩 기반 유사도 검색으로 교체할 예정입니다.

def search_faq(query, faq_data, top_k=3):
    """키워드 매칭 점수로 관련 FAQ를 검색합니다.

    점수 산정 방식 (가중치):
      - FAQ keywords 목록에 있는 단어가 query에 있으면 +2점
      - query의 단어가 FAQ question 텍스트에 있으면 +1점
      - FAQ category 이름이 query에 직접 언급되면 +3점

    Args:
        query   : 사용자 질문 문자열
        faq_data: FAQ 딕셔너리 리스트
        top_k   : 반환할 최대 FAQ 개수 (기본값 3)

    Returns:
        score > 0인 항목을 점수 내림차순으로 정렬한 FAQ 리스트
    """
    scores = []

    # 질문에서 의미 있는 단어(2자 이상)만 추출하여 비교에 사용
    query_words = [
        w for w in query.replace("?", "").replace(".", "").split()
        if len(w) >= 2
    ]
    # print(query_words)
    for faq in faq_data:
        score = 0

        # ① 키워드 매칭: FAQ 작성자가 미리 정의한 핵심 단어 (가중치 높음)
        for keyword in faq["keywords"]:
            if keyword in query:
                score += 2

        # ② 질문 텍스트 매칭: 사용자 질문 단어가 FAQ 질문에 등장하면
        for word in query_words:
            if word in faq["question"]:
                score += 1

        # ③ 카테고리 직접 언급: 명시적으로 카테고리명을 말한 경우 (가중치 최高)
        if faq["category"] in query:
            score += 3

        scores.append((score, faq))

    # 점수 내림차순 정렬 후 점수 > 0인 상위 top_k 반환
    scores.sort(key=lambda x: x[0], reverse=True)
    return [faq for score, faq in scores[:top_k] if score > 0]


# ── 검색 정확도 테스트 ────────────────────────────────────────────
# 예상 FAQ ID와 실제 검색 결과 1위가 일치하는지 확인
# ⭐️키워드는 일치하여 context에 포함되지만, 전혀 관계 없는 질문은 LLM이 거를 수 있나? - 있음
# but, 좀 더 정확히 예상 답변을 솎아낼 수 있다면...?
print("🔍 FAQ 검색 테스트:\n")
hit = 0
for test in SAMPLE_TEST_QUERIES:
    results = search_faq(test["query"], SAMPLE_FAQ_DATA)
    matched_id = results[0]["id"] if results else "없음"
    is_correct = matched_id == test["expected_faq_id"]
    status = "✅" if is_correct else "⚠️"
    if is_correct:
        hit += 1
    print(f"{status} 질문: {test['query']}")
    print(f"   예상: {test['expected_faq_id']} | 검색결과: {matched_id} ({len(results)}개 매칭)")
    if results:
        print(f"   1위 FAQ: {results[0]['question']}")
    print()

print(f"📈 검색 정확도: {hit}/{len(SAMPLE_TEST_QUERIES)} ({hit/len(SAMPLE_TEST_QUERIES)*100:.0f}%)")

🔍 FAQ 검색 테스트:

✅ 질문: 청약통장 가입하려면 어떻게 해요?
   예상: FAQ001 | 검색결과: FAQ001 (3개 매칭)
   1위 FAQ: 주택청약종합저축이란 무엇인가요?

✅ 질문: 1순위 되려면 뭐가 필요해요?
   예상: FAQ004 | 검색결과: FAQ004 (1개 매칭)
   1위 FAQ: 청약통장 1순위 조건은 무엇인가요?

✅ 질문: 신혼부부 특공 자격이 궁금해요
   예상: FAQ010 | 검색결과: FAQ010 (2개 매칭)
   1위 FAQ: 신혼부부 특별공급 조건은 무엇인가요?

⚠️ 질문: 가점이 높으면 유리한가요?
   예상: FAQ013 | 검색결과: FAQ010 (1개 매칭)
   1위 FAQ: 신혼부부 특별공급 조건은 무엇인가요?

✅ 질문: 당첨되면 어떻게 확인해요?
   예상: FAQ017 | 검색결과: FAQ017 (2개 매칭)
   1위 FAQ: 당첨자 발표는 어떻게 확인하나요?

⚠️ 질문: 역대 당첨자 중 최고 가점은?
   예상: FAQ013 | 검색결과: FAQ010 (2개 매칭)
   1위 FAQ: 신혼부부 특별공급 조건은 무엇인가요?

⚠️ 질문: 청약이 당첨되면 로또겠죠?
   예상:  | 검색결과: 없음 (0개 매칭)

📈 검색 정확도: 4/7 (57%)


---
## 사이클 4: 검색 결과 + LLM 답변 생성

검색된 FAQ를 system prompt에 넣어 답변을 생성하는 `ask_faq(question, faq_data, client)` 함수를 만드세요. 답변과 함께 참고한 FAQ 목록도 반환하세요.

In [42]:
# ── 사이클 4: 검색 결과 + LLM 답변 생성 ──────────────────────────
# RAG(Retrieval-Augmented Generation)의 핵심 아이디어:
#   1. 검색(Retrieve): 질문과 관련된 문서를 DB에서 찾는다
#   2. 증강(Augment) : 찾은 문서를 프롬프트에 삽입(주입)한다
#   3. 생성(Generate): LLM이 주입된 문서를 참고해 답변을 생성한다
#
# → 이 방식 덕분에 LLM이 학습 데이터에 없는 최신/특정 정보도 답변 가능!

def ask_faq(question, faq_data, client):
    """FAQ 검색 결과를 시스템 프롬프트에 주입하여 LLM 답변을 생성합니다.

    Returns:
        dict:
          answer         (str) : LLM이 생성한 답변
          referenced_faqs(list): 참고한 FAQ ID 목록
          faq_count      (int) : 참고한 FAQ 개수
    """
    # ── Step 1: 관련 FAQ 검색 ──────────────────────────────────────
    relevant_faqs = search_faq(question, faq_data, top_k=3)

    # ── Step 2: 검색된 FAQ를 문자열 컨텍스트로 변환 ──────────────────
    # "\n\n".join([...]): 각 FAQ 사이에 빈 줄을 넣어 가독성 확보
    if relevant_faqs:
        context = "\n\n".join([
            f"[{faq['category']}] Q: {faq['question']}\nA: {faq['answer']}"
            for faq in relevant_faqs
        ])
    else:
        # 검색 결과 없을 때도 LLM이 처리할 수 있도록 안내 문구 삽입
        context = "관련 FAQ를 찾을 수 없습니다."

    # ── Step 3: 컨텍스트를 system 메시지에 주입하여 LLM 호출 ─────────
    # f-string으로 context를 system 메시지 안에 동적으로 삽입
    response = client.chat.completions.create(
        model="gpt-4o-mini",
        messages=[
            {
                "role": "system",
                "content": (
                    "당신은 주택청약 전문 상담원입니다.\n"
                    "아래 FAQ 데이터를 참고하여 질문에 친절하게 답변해 주세요.\n"
                    "데이터에 없는 내용은 '청약홈(www.applyhome.co.kr) 또는 전문 상담을 통해 확인하세요'라고 안내하세요.\n\n"
                    f"[FAQ 데이터]\n{context}"  # ← 검색된 FAQ가 여기에 주입됨
                )
            },
            {"role": "user", "content": question}
        ],
        temperature=0
    )

    return {
        "answer": response.choices[0].message.content,
        "referenced_faqs": [faq["id"] for faq in relevant_faqs],
        "faq_count": len(relevant_faqs)
    }

# ── Yes 테스트 ────────────────────────────────────────────────────────
result = ask_faq("신혼부부 가점에 대해 알려주세요", SAMPLE_FAQ_DATA, client)
print(f"💬 답변:\n{result['answer']}")
print(f"\n📎 참고 FAQ: {result['referenced_faqs']} ({result['faq_count']}개)")
print()

result = ask_faq("아이에게 청약통장을 만들어 주고 싶어요", SAMPLE_FAQ_DATA, client)
print(f"💬 답변:\n{result['answer']}")
print(f"\n📎 참고 FAQ: {result['referenced_faqs']} ({result['faq_count']}개)")
print('────────────────────────────────────────────────────────')

# ── No 테스트 ────────────────────────────────────────────────────────
result = ask_faq("청 약 점수에 대해 알 려주세요", SAMPLE_FAQ_DATA, client)
print(f"💬 답변:\n{result['answer']}")
print(f"\n📎 참고 FAQ: {result['referenced_faqs']} ({result['faq_count']}개)")
print()

result = ask_faq("주택청약은 몇 세까지인가요?", SAMPLE_FAQ_DATA, client)
print(f"💬 답변:\n{result['answer']}")
print(f"\n📎 참고 FAQ: {result['referenced_faqs']} ({result['faq_count']}개)")

💬 답변:
신혼부부 특별공급의 가점은 다음과 같은 요소에 따라 결정됩니다:

1. **혼인기간**: 혼인기간이 짧을수록 가점이 높습니다. 즉, 혼인기간이 7년 이내인 경우에 해당합니다.
2. **자녀 수**: 자녀가 많을수록 가점이 높습니다. 자녀가 1명 이상일 경우 추가 가점을 받을 수 있습니다.

이 외에도 신혼부부 특별공급의 기본 조건은 무주택 세대주이며, 도시근로자 월평균소득이 100~140% 이내여야 하고, 전용면적이 85㎡ 이하인 주택에 해당해야 합니다.

더 자세한 내용은 청약홈(www.applyhome.co.kr) 또는 전문 상담을 통해 확인하세요.

📎 참고 FAQ: ['FAQ010', 'FAQ009'] (2개)

💬 답변:
아이에게 청약통장을 만들어 주는 것은 좋은 선택입니다! 주택청약종합저축은 국민주택과 민영주택 모두에 청약할 수 있는 만능 통장으로, 매월 2만원에서 50만원까지 자유롭게 납입할 수 있습니다. 가입 후 일정 기간이 경과하면 청약 자격이 부여되므로, 아이가 성인이 되었을 때 주택 청약에 유리한 조건을 갖출 수 있습니다.

청약통장 개설에 대한 자세한 사항은 청약홈(www.applyhome.co.kr) 또는 전문 상담을 통해 확인하세요.

📎 참고 FAQ: ['FAQ001', 'FAQ004'] (2개)
────────────────────────────────────────────────────────
💬 답변:
청약 점수는 주택청약을 신청할 때, 신청자의 자격을 평가하기 위해 부여되는 점수입니다. 이 점수는 주택청약의 당첨 확률에 영향을 미치며, 보통 청약 통장 가입 기간, 납입 횟수, 세대주 여부, 무주택 기간 등 여러 요소에 따라 결정됩니다. 

자세한 점수 산정 기준이나 방법에 대해서는 청약홈(www.applyhome.co.kr) 또는 전문 상담을 통해 확인하세요.

📎 참고 FAQ: [] (0개)

💬 답변:
주택청약에 대한 구체적인 연령 제한은 없습니다. 하지만 청약을 신청하기 위해서는 만 19세 이상이어야 합니다. 

---
## 사이클 5: PromptTemplate

`ChatPromptTemplate`으로 `{context}`와 `{question}` 변수를 사용하는 FAQ 답변용 프롬프트를 만들고, 카테고리 분류용 프롬프트도 하나 더 만들어서 각각 테스트하세요.

In [25]:
# ── 사이클 5: PromptTemplate ──────────────────────────────────────
# 사이클 4에서 f-string으로 하드코딩했던 프롬프트를
# ChatPromptTemplate으로 재사용 가능한 형태로 분리합니다.
#
# ChatPromptTemplate의 장점:
#   - 프롬프트와 비즈니스 로직을 분리 → 유지보수 용이
#   - {변수명} 형태로 동적 값 주입
#   - LCEL 체인(사이클 6)에 바로 연결 가능: prompt | llm | parser

from langchain_core.prompts import ChatPromptTemplate

# ── 프롬프트 1: FAQ 답변용 ────────────────────────────────────────
# {context}: 검색된 FAQ 텍스트가 주입될 자리
# {question}: 사용자 질문이 주입될 자리
faq_prompt = ChatPromptTemplate.from_messages([
    ("system",
     """당신은 주택청약 전문 상담원입니다.
아래 FAQ 데이터를 참고하여 친절하고 정확하게 답변해 주세요.
답변 첫 줄에 반드시 '📂 카테고리: [카테고리명]' 형식으로 카테고리를 표시해 주세요.
데이터에 없는 내용은 '청약홈(www.applyhome.co.kr) 또는 전문 상담을 통해 확인하세요'라고 안내하세요.

[FAQ 데이터]
{context}"""),
    ("human", "{question}")
])

# ── 프롬프트 2: 카테고리 분류용 ──────────────────────────────────
# 질문을 받아 정해진 카테고리 중 하나로 분류
# "반드시 카테고리 이름만" → LLM이 쓸데없는 설명을 붙이지 않도록 강제
category_prompt = ChatPromptTemplate.from_messages([
    ("system",
     """아래 카테고리 중 질문에 가장 적합한 것을 하나만 선택하세요.
카테고리: 청약통장, 청약자격, 특별공급, 일반공급, 당첨/계약, 기타
반드시 카테고리 이름만 답하세요. 설명을 추가하지 마세요."""),
    ("human", "{question}")
])

# ── 테스트 ────────────────────────────────────────────────────────
test_question = "청약 1순위 조건이 뭔가요?"

# 검색된 FAQ를 컨텍스트 문자열로 변환
test_faqs = search_faq(test_question, SAMPLE_FAQ_DATA)
# print(f'test_faqs: {len(test_faqs)}')

test_context = "\n\n".join([
    f"[{f['category']}] Q: {f['question']}\nA: {f['answer']}"
    for f in test_faqs
])

# prompt | llm : 프롬프트를 완성한 뒤 LLM에 전달 (파이프라인 미리 맛보기)
faq_result = (faq_prompt | llm).invoke({"context": test_context, "question": test_question})
cat_result = (category_prompt | llm).invoke({"question": test_question})

print("📝 FAQ 답변 프롬프트 테스트:")
print(f"Q: {test_question}")
print(f"A: {faq_result.content}")
print(f"\n🏷️  카테고리 분류 프롬프트 테스트:")
print(f"Q: {test_question} → 분류: {cat_result.content}")

📝 FAQ 답변 프롬프트 테스트:
Q: 청약 1순위 조건이 뭔가요?
A: 청약 1순위 조건은 주택 유형에 따라 다릅니다.

1) 민영주택: 수도권 12개월, 비수도권 6개월 + 예치금
2) 국민주택: 수도권 12개월(24회), 비수도권 6개월(12회)
3) 투기과열지구: 2년, 24회 납입

이 조건을 충족하면 청약 1순위로 신청할 수 있습니다.

🏷️  카테고리 분류 프롬프트 테스트:
Q: 청약 1순위 조건이 뭔가요? → 분류: 청약자격


---
## 사이클 6: LCEL 체인

`prompt | llm | StrOutputParser()` 패턴으로 FAQ 답변 체인(`faq_chain`)을 만들고, 질문 2개로 테스트하세요. `.stream()`으로 스트리밍 출력도 해보세요.

In [43]:
# ── 사이클 6: LCEL 체인 ───────────────────────────────────────────
# ⭐️LCEL(LangChain Expression Language):
#   파이프 연산자(|)로 컴포넌트를 연결해 데이터 흐름을 표현하는 문법
#
#   prompt | llm | parser
#     ↓       ↓      ↓
#   입력 포맷  LLM 호출  문자열 추출
#
# StrOutputParser: LLM 응답 객체(AIMessage)에서 .content(문자열)만 꺼냄
# → 없으면 AIMessage 객체 그대로 반환되어 이후 처리가 불편

from langchain_core.output_parsers import StrOutputParser

# faq_chain: 사이클 7~10에서 계속 재사용되는 핵심 체인
faq_chain = faq_prompt | llm | StrOutputParser()

def get_context(query):
    """질문으로 FAQ를 검색하여 프롬프트에 주입할 컨텍스트 문자열을 반환합니다."""
    faqs = search_faq(query, SAMPLE_FAQ_DATA, top_k=3)
    if not faqs:
        return "관련 FAQ를 찾을 수 없습니다."
    return "\n\n".join([
        f"[{f['category']}] Q: {f['question']}\nA: {f['answer']}"
        for f in faqs
    ])

# ── 테스트 1: invoke (동기 호출, 전체 답변을 한 번에 반환) ────────
q1 = "신혼부부 특공 조건이 뭔가요?"
result1 = faq_chain.invoke({"context": get_context(q1), "question": q1})
print(f"📝 [invoke] Q: {q1}")
print(f"A: {result1}\n")

# ── 테스트 2: stream (스트리밍 호출, 토큰 단위로 실시간 출력) ─────
# stream: 답변이 생성되는 즉시 청크(chunk) 단위로 받아 출력
# end="", flush=True → 줄바꿈 없이 이어서 출력 (타이핑 효과)
q2 = "무주택자 기준이 뭔가요?"
print(f"📡 [stream] Q: {q2}")
print("A: ", end="")
for chunk in faq_chain.stream({"context": get_context(q2), "question": q2}):
    print(chunk, end="", flush=True)
print('\n')  # 마지막 줄바꿈

# ── 테스트 3: .invoke() 호출은 완전히 독립적 ❌ ────────
q3 = "내가 처음했던 질문이 뭐였지?"
result3 = faq_chain.invoke({"context": get_context(q3), "question": q3})
print(f"📝 [invoke] Q: {q3}")
print(f"A: {result3}\n")

📝 [invoke] Q: 신혼부부 특공 조건이 뭔가요?
A: 신혼부부 특별공급의 조건은 다음과 같습니다:

1) 혼인기간 7년 이내 무주택 세대주
2) 소득: 도시근로자 월평균소득 100~140%
3) 전용면적 85㎡ 이하
4) 혼인기간이 짧을수록, 자녀가 많을수록 가점이 높습니다.
5) 예비 신혼부부도 신청이 가능합니다.

더 궁금한 사항이 있으시면 청약홈(www.applyhome.co.kr) 또는 전문 상담을 통해 확인하세요.

📡 [stream] Q: 무주택자 기준이 뭔가요?
A: 무주택자 기준은 본인과 세대원 모두 주택을 소유하지 않을 때 무주택자로 간주됩니다. 단, 다음과 같은 예외가 있습니다: 60세 이상 직계존속이 소유한 주택, 20㎡ 이하의 소형주택, 상속 후 3개월 이내에 처분해야 하는 주택은 무주택자로 인정됩니다. 또한, 분양권이나 입주권도 주택 수에 포함됩니다.

📝 [invoke] Q: 내가 처음했던 질문이 뭐였지?
A: 죄송하지만, 이전 질문 내용을 확인할 수 없습니다. 궁금한 점이 있으시면 다시 질문해 주시면 친절하게 답변해 드리겠습니다.



---
## 사이클 7: 검색

질문을 넣으면 자동으로 FAQ 검색 → 답변 생성하는 `rag_chain`을 만드세요. `SAMPLE_TEST_QUERIES` 5개로 테스트하세요.

In [34]:
# ── 사이클 7: RAG 체인 ────────────────────────────────────────────
# 사이클 6까지는 get_context()를 수동으로 호출해야 했습니다.
# 이제 RunnableLambda로 검색 단계를 체인에 포함시켜
# ⭐️질문 하나만 넣으면 자동으로 검색→생성이 동작하는 RAG 체인을 만듭니다.
#
# RunnableLambda: 일반 파이썬 함수를 LCEL 체인에 연결할 수 있게 감싸주는 래퍼
# ⭐️전체 흐름: query(str) → build_rag_input(dict) → faq_chain(str)

from langchain_core.runnables import RunnableLambda

def build_rag_input(query):
  """질문 문자열을 받아 faq_chain이 필요한 딕셔너리 형태로 변환합니다.

  faq_chain 입력 형식: {"context": str, "question": str}
  """
  relevant_faqs = search_faq(query, SAMPLE_FAQ_DATA, top_k=3)

  if relevant_faqs:
      # 검색된 FAQ를 프롬프트에 삽입할 컨텍스트 문자열로 조합
      # context는 검색 결과를 LLM이 읽을 수 있는 텍스트 형태로 변환한 것
      # RAG의 "A(Augment, 증강)" 단계!!!
      context = "\n\n".join([
          f"C: [{faq['category']}]\n" +
          f"Q: {faq['question']}\nA: {faq['answer']}"
          for faq in relevant_faqs
      ])
  else:
      context = "관련 FAQ를 찾을 수 없습니다."

  # faq_chain(faq_prompt)의 입력 변수 {context}, {question}에 맞게 반환
  return {"context": context, "question": query}

# ⭐️⭐️⭐️rag_chain: query(문자열) 하나만 넣으면 검색+생성이 자동 수행됨
rag_chain = RunnableLambda(build_rag_input) | faq_chain

# ── SAMPLE_TEST_QUERIES로 테스트 ─────────────────────────────
print("🔍 RAG 체인 테스트 (6개 질의):\n")
for test in SAMPLE_TEST_QUERIES:
    answer = rag_chain.invoke(test["query"])  # 이제 질문 문자열만 전달하면 됨!
    print(f"Q: {test['query']}")
    print(f"A: {answer[:100]}...\n")

# 카테고리가 답변에 안 나오는 이유
# context에 카테고리(C: [청약통장])를 넣었지만, LLM한테 "카테고리를 출력하라"는 지시가 없었다.
# LLM은 context를 참고 자료로만 쓰고, 어떤 형식으로 답할지는 프롬프트 지시에 따르미.
# 앞서 만들었던 category_prompt를 같이 사용하고 싶다면??? - RunnableParallel⭐️

🔍 RAG 체인 테스트 (6개 질의):

Q: 청약통장 가입하려면 어떻게 해요?
A: 청약통장 가입 방법에 대한 정보는 청약홈(www.applyhome.co.kr) 또는 전문 상담을 통해 확인하세요....

Q: 1순위 되려면 뭐가 필요해요?
A: 1순위 조건은 주택 유형에 따라 다릅니다. 

1) 민영주택: 수도권은 12개월, 비수도권은 6개월의 가입 기간이 필요하며, 예치금도 요구됩니다.
2) 국민주택: 수도권은 12개월...

Q: 신혼부부 특공 자격이 궁금해요
A: 신혼부부 특별공급의 자격 조건은 다음과 같습니다:

1) 혼인기간 7년 이내의 무주택 세대주
2) 소득: 도시근로자 월평균소득 100~140% 이내
3) 전용면적 85㎡ 이하의 주...

Q: 가점이 높으면 유리한가요?
A: 네, 가점이 높으면 주택청약에서 유리합니다. 특히 신혼부부 특별공급의 경우, 혼인기간이 짧을수록, 자녀가 많을수록 가점이 높아지기 때문에, 이러한 조건을 충족하면 더 좋은 결과를 ...

Q: 당첨되면 어떻게 확인해요?
A: 당첨자 발표는 다음과 같이 확인하실 수 있습니다:

1) 청약홈(www.applyhome.co.kr) 접속
2) 당첨자 조회 메뉴 클릭
3) 문자 알림 서비스 신청 가능

당첨 후...

Q: 청약이 당첨되면 로또겠죠?
A: 청약이 당첨되는 것은 로또와는 다릅니다. 청약은 주택을 구매하기 위한 제도로, 일정한 조건을 충족한 후 신청하여 당첨되는 방식입니다. 로또는 무작위로 번호를 선택하여 당첨 여부를 ...



---
## 사이클 8: 에러 처리

빈 입력, 500자 초과, 숫자만 입력 등을 검증하고 `try/except`로 API 오류를 처리하는 `safe_ask(question, rag_chain)` 함수를 만드세요. 정상/에러 케이스 6가지 이상 테스트하세요.

In [44]:
# ── 사이클 8: 에러 처리 ───────────────────────────────────────────
# 실서비스에서 발생할 수 있는 두 종류의 오류를 처리합니다.
#
# 1. ⭐️입력 검증 오류 (API 호출 전에 차단 → 비용 절감)
#    - 빈 입력, 너무 짧은 질문, 너무 긴 질문, 숫자만 입력 등
#
# 2. ⭐️API 런타임 오류 (try/except로 처리)
#    - 네트워크 불안정, API 키 만료, 요청 한도 초과 등

import time

def validate_input(question):
    """입력 유효성 검사.

    Returns:
        None  : 유효한 입력 (오류 없음)
        str   : 사용자에게 보여줄 오류 메시지
    """
    # 빈 입력 또는 공백만 있는 경우
    if not question or not question.strip():
        return "❌ 질문을 입력해 주세요."
    # 너무 짧은 입력 (의미 있는 질문이 될 수 없음)
    if len(question.strip()) < 5:
        return "❌ 질문이 너무 짧습니다. 5자 이상 입력해 주세요."
    # 500자 초과 (토큰 낭비 및 프롬프트 오염 방지)
    if len(question) > 500:
        return f"❌ 질문이 너무 깁니다. 500자 이하로 입력해 주세요. (현재: {len(question)}자)"
    # 숫자만 입력한 경우 (의미 없는 질문)
    if question.strip().isdigit():
        return "❌ 숫자만 입력할 수 없습니다. 청약 관련 질문을 입력해 주세요."
    return None  # 모든 검증 통과


def safe_ask(question, rag_chain):
    """입력 검증 + RAG 체인 실행 + 예외 처리를 모두 포함한 안전한 답변 함수.

    Returns:
        dict:
          success    (bool): 정상 처리 여부
          answer     (str) : 답변 또는 오류 메시지
          elapsed_ms (int) : 소요 시간 (밀리초)
    """
    # ── Step 1: 입력 검증 (API 호출 전 차단) ─────────────────────
    error = validate_input(question)
    if error:
        return {"success": False, "answer": error, "elapsed_ms": 0}

    # ── Step 2: RAG 체인 실행 (예외 발생 가능 구간) ───────────────
    # time.perf_counter(): 고해상도 타이머 (응답 시간 측정용)
    start = time.perf_counter()
    try:
        answer = rag_chain.invoke(question.strip())
        elapsed_ms = int((time.perf_counter() - start) * 1000)
        return {"success": True, "answer": answer, "elapsed_ms": elapsed_ms}
    except Exception as e:
        # 실제 오류는 서버 로그에 기록하되, 사용자에게는 친절한 메시지만 노출
        elapsed_ms = int((time.perf_counter() - start) * 1000)
        print(f"[ERROR] {type(e).__name__}: {e}")  # 개발자용 로그
        return {
            "success": False,
            "answer": "⚠️ 오류가 발생했습니다. 잠시 후 다시 시도해 주세요.",
            "elapsed_ms": elapsed_ms
        }


# ── 테스트 케이스 6가지 ───────────────────────────────────────────
# (질문, 케이스 설명) 튜플 리스트
test_cases = [
    ("청약 1순위 조건이 뭔가요?",            "✅ 정상 질문"),
    ("",                               "❌ 빈 입력"),
    ("네",                              "❌ 너무 짧음 (2자)"),
    ("1234567890",                     "❌ 숫자만 입력"),
    ("가" * 501,                        "❌ 500자 초과"),
    ("신혼부부 특별공급 소득 기준이 궁금합니다.", "✅ 정상 질문 2"),
]

print("🧪 safe_ask 테스트:\n")
for question, case_name in test_cases:
    result = safe_ask(question, rag_chain)
    # 30자 초과 질문은 잘라서 표시
    display_q = (question[:30] + "...") if len(question) > 30 else question
    print(f"[{case_name}]")
    print(f"  입력: '{display_q}'")
    print(f"  답변: {result['answer'][:80]}")
    print(f"  응답시간: {result['elapsed_ms']}ms\n")

🧪 safe_ask 테스트:

[✅ 정상 질문]
  입력: '청약 1순위 조건이 뭔가요?'
  답변: 청약 1순위 조건은 주택 유형에 따라 다릅니다.

1) 민영주택: 수도권 12개월, 비수도권 6개월 + 예치금
2) 국민주택: 수도권 12개월(
  응답시간: 2371ms

[❌ 빈 입력]
  입력: ''
  답변: ❌ 질문을 입력해 주세요.
  응답시간: 0ms

[❌ 너무 짧음 (2자)]
  입력: '네'
  답변: ❌ 질문이 너무 짧습니다. 5자 이상 입력해 주세요.
  응답시간: 0ms

[❌ 숫자만 입력]
  입력: '1234567890'
  답변: ❌ 숫자만 입력할 수 없습니다. 청약 관련 질문을 입력해 주세요.
  응답시간: 0ms

[❌ 500자 초과]
  입력: '가가가가가가가가가가가가가가가가가가가가가가가가가가가가가가...'
  답변: ❌ 질문이 너무 깁니다. 500자 이하로 입력해 주세요. (현재: 501자)
  응답시간: 0ms

[✅ 정상 질문 2]
  입력: '신혼부부 특별공급 소득 기준이 궁금합니다.'
  답변: 신혼부부 특별공급의 소득 기준은 도시근로자 월평균소득의 100%에서 140% 사이입니다. 즉, 이 범위 내의 소득을 가진 무주택 세대주가 신청할
  응답시간: 2482ms



---
## 사이클 9: Gradio 채팅 UI

`gr.ChatInterface`로 지금까지 만든 RAG 체인을 웹 채팅 UI로 만드세요. 제목, 설명, 예시 질문 5개를 설정하세요.

In [45]:
# ── 사이클 9: Gradio 채팅 UI ──────────────────────────────────────
# ⭐️ Gradio: 파이썬 함수를 즉시 웹 UI로 만들어주는 라이브러리
# gr.ChatInterface: 채팅 UI 전용 컴포넌트 (history 관리 자동)
#
# 콜백 함수 시그니처 규칙:
#   fn(message: str, history: list[tuple[str, str]]) -> str
#     message : 현재 사용자 입력
#     history : [(user1, bot1), (user2, bot2), ...] 형태의 이전 대화 (현재는 미사용)
#     return  : 봇의 답변 문자열

import gradio as gr

def chat_response(message, history):
    """Gradio ChatInterface 콜백 함수. safe_ask로 입력 검증 후 rag_chain 실행."""
    result = safe_ask(message, rag_chain)
    return result["answer"]


demo = gr.ChatInterface(
    fn=chat_response,
    title="🏠 주택청약 FAQ 챗봇",
    description="주택청약에 관한 궁금한 점을 질문해 주세요. AI가 FAQ를 바탕으로 답변합니다.",
    # examples: 클릭 한 번으로 입력창에 채워지는 예시 질문들
    examples=[
        "청약통장은 어떻게 가입하나요?",
        "청약 1순위 조건이 뭔가요?",
        "신혼부부 특공 자격이 궁금해요",
        "무주택자 기준을 알려주세요",
        "당첨되면 어디서 확인하나요?",
    ],
    theme=gr.themes.Soft(),   # 부드러운 색상 테마 적용
)

# share=True: 외부에서 접속 가능한 공개 URL 생성 (Colab 필수)
demo.launch(share=True)

/usr/local/lib/python3.12/dist-packages/gradio/chat_interface.py:347: UserWarning: The 'tuples' format for chatbot messages is deprecated and will be removed in a future version of Gradio. Please set type='messages' instead, which uses openai-style 'role' and 'content' keys.
  self.chatbot = Chatbot(


Colab notebook detected. To show errors in colab notebook, set debug=True in launch()
* Running on public URL: https://cca95fe6ea36b87868.gradio.live

This share link expires in 1 week. For free permanent hosting and GPU upgrades, run `gradio deploy` from the terminal in the working directory to deploy to Hugging Face Spaces (https://huggingface.co/spaces)


---
## 사이클 10: 최종 통합 테스트

전체 파이프라인(입력 검증 → 검색 → 답변 생성)을 하나의 함수로 정리하고, 10개 질문으로 테스트하세요. 각 질문의 응답 시간, 참고 FAQ 수를 포함한 결과표를 출력하세요. Gradio UI도 최종 버전으로 만드세요.

In [46]:
# ── 사이클 10: 최종 통합 테스트 ─────────────────────────────────
import gradio as gr
import time

def full_pipeline(question):
    """입력 검증 → FAQ 검색 → LLM 답변 생성을 하나로 묶은 최종 파이프라인.

    safe_ask와의 차이:
      - referenced_faqs, faq_count 등 검색 메타데이터 포함
      - 성능 측정 및 결과표 출력에 활용

    Returns:
        dict: success, answer, referenced_faqs, faq_count, elapsed_ms
    """
    start = time.perf_counter()

    # ── Step 1: 입력 검증 ─────────────────────────────────────────
    error = validate_input(question)
    if error:
        return {"success": False, "answer": error,
                "referenced_faqs": [], "faq_count": 0, "elapsed_ms": 0}

    # ── Step 2: FAQ 검색 (메타데이터 보존을 위해 직접 호출) ────────
    relevant_faqs = search_faq(question.strip(), SAMPLE_FAQ_DATA, top_k=3)
    faq_ids = [faq["id"] for faq in relevant_faqs]

    # ── Step 3: 컨텍스트 생성 + LLM 호출 ─────────────────────────
    try:
        context = (
            "\n\n".join([
                f"[{faq['category']}] Q: {faq['question']}\nA: {faq['answer']}"
                for faq in relevant_faqs
            ])
            if relevant_faqs
            else "관련 FAQ를 찾을 수 없습니다."
        )
        # faq_chain은 사이클 6에서 만든 LCEL 체인 재사용
        answer = faq_chain.invoke({"context": context, "question": question.strip()})
        elapsed_ms = int((time.perf_counter() - start) * 1000)
        return {
            "success": True,
            "answer": answer,
            "referenced_faqs": faq_ids,
            "faq_count": len(relevant_faqs),
            "elapsed_ms": elapsed_ms
        }
    except Exception as e:
        elapsed_ms = int((time.perf_counter() - start) * 1000)
        print(f"[ERROR] {type(e).__name__}: {e}")
        return {
            "success": False,
            "answer": "⚠️ 오류가 발생했습니다. 잠시 후 다시 시도해 주세요.",
            "referenced_faqs": [],
            "faq_count": 0,
            "elapsed_ms": elapsed_ms
        }


# ─────────────────────────────────────────────────────────────────
# 10개 질문 성능 결과표 출력
# ─────────────────────────────────────────────────────────────────
test_questions = [
    "주택청약종합저축이란 무엇인가요?",
    "청약 1순위 조건이 뭔가요?",
    "신혼부부 특공 자격이 궁금해요",
    "가점제와 추첨제 차이가 뭔가요?",
    "무주택자 기준을 알려주세요",
    "당첨 확인은 어디서 해요?",
    "재당첨 제한이 뭔가요?",
    "특별공급 종류가 뭐가 있나요?",
    "청약홈 사용법을 알려주세요",
    "다자녀 특공 조건이 궁금합니다",
]

print("=" * 72)
print(f"{'질문':<34} {'참고FAQ':<16} {'응답시간':>8} {'결과':>4}")
print("=" * 72)

total_ms = 0
for q in test_questions:
    result = full_pipeline(q)
    status = "✅" if result["success"] else "❌"
    short_q = (q[:31] + "...") if len(q) > 31 else q
    # 참고 FAQ가 없으면 "없음" 표시
    faq_refs = ", ".join(result["referenced_faqs"]) if result["referenced_faqs"] else "없음"
    total_ms += result["elapsed_ms"]
    print(f"{short_q:<34} {faq_refs:<16} {result['elapsed_ms']:>6}ms {status:>4}")

print("=" * 72)
print(f"평균 응답시간: {total_ms // len(test_questions)}ms")


# ─────────────────────────────────────────────────────────────────
# 최종 Gradio UI (v1.0)
# ─────────────────────────────────────────────────────────────────
def final_chat(message, history):
    """최종 Gradio 콜백. 답변 하단에 참고 FAQ와 응답시간 메타데이터를 추가."""
    result = full_pipeline(message)
    if result["success"] and result["referenced_faqs"]:
        # 답변 본문 + 참고 FAQ ID + 응답시간을 함께 표시
        footer = f"\n\n📎 참고 FAQ: {', '.join(result['referenced_faqs'])} | ⏱️ {result['elapsed_ms']}ms"
        return result["answer"] + footer
    return result["answer"]


final_demo = gr.ChatInterface(
    fn=final_chat,
    title="🏠 주택청약 FAQ 챗봇 v1.0",
    description=(
        "주택청약에 관한 궁금한 점을 질문해 주세요.\n"
        "AI가 FAQ를 바탕으로 정확한 정보를 안내합니다.\n"
        "📌 출처: 한국부동산원 청약홈(www.applyhome.co.kr)"
    ),
    examples=[
        "청약통장 가입하려면 어떻게 해요?",
        "1순위 되려면 뭐가 필요해요?",
        "신혼부부 특공 자격이 궁금해요",
        "가점이 높으면 유리한가요?",
        "당첨되면 어떻게 확인해요?",
    ],
    theme=gr.themes.Soft(),
    chatbot=gr.Chatbot(height=450), # 채팅창 높이 고정
)

final_demo.launch(share=True)

질문                                 참고FAQ                응답시간   결과
주택청약종합저축이란 무엇인가요?                  FAQ001, FAQ004, FAQ005   2629ms    ✅
청약 1순위 조건이 뭔가요?                    FAQ004, FAQ001, FAQ005   2451ms    ✅
신혼부부 특공 자격이 궁금해요                   FAQ010, FAQ009     2725ms    ✅
가점제와 추첨제 차이가 뭔가요?                  FAQ013, FAQ010     3687ms    ✅
무주택자 기준을 알려주세요                     FAQ006, FAQ005     2248ms    ✅
당첨 확인은 어디서 해요?                     FAQ017, FAQ020     1360ms    ✅
재당첨 제한이 뭔가요?                       FAQ020             2844ms    ✅
특별공급 종류가 뭐가 있나요?                   FAQ009, FAQ010     2391ms    ✅
청약홈 사용법을 알려주세요                     FAQ023, FAQ017     2588ms    ✅
다자녀 특공 조건이 궁금합니다                   FAQ009             2264ms    ✅
평균 응답시간: 2518ms


/tmp/ipykernel_662/487511045.py:129: UserWarning: You have not specified a value for the `type` parameter. Defaulting to the 'tuples' format for chatbot messages, but this is deprecated and will be removed in a future version of Gradio. Please set type='messages' instead, which uses openai-style dictionaries with 'role' and 'content' keys.
  chatbot=gr.Chatbot(height=450),  # 채팅창 높이 고정
/tmp/ipykernel_662/487511045.py:129: DeprecationWarning: The default value of 'allow_tags' in gr.Chatbot will be changed from False to True in Gradio 6.0. You will need to explicitly set allow_tags=False if you want to disable tags in your chatbot.
  chatbot=gr.Chatbot(height=450),  # 채팅창 높이 고정
/usr/local/lib/python3.12/dist-packages/gradio/chat_interface.py:330: UserWarning: The gr.ChatInterface was not provided with a type, so the type of the gr.Chatbot, 'tuples', will be used.
  warnings.warn(


Colab notebook detected. To show errors in colab notebook, set debug=True in launch()
* Running on public URL: https://e3df89a30cd49db58a.gradio.live

This share link expires in 1 week. For free permanent hosting and GPU upgrades, run `gradio deploy` from the terminal in the working directory to deploy to Hugging Face Spaces (https://huggingface.co/spaces)


---
## Weekend 1 완료! 🎉

| 사이클 | 구현 내용 | 핵심 개념 |
|--------|-----------|----------|
| 1 | 첫 API 호출 | `client.chat.completions.create`, system role |
| 2 | FAQ 데이터 탐색 | `Counter`, 리스트 컴프리헨션 |
| 3 | 키워드 검색 함수 | 점수 기반 정렬, `search_faq` |
| 4 | 검색 + LLM 통합 | 컨텍스트 주입(RAG 원리), `ask_faq` |
| 5 | PromptTemplate | `ChatPromptTemplate`, 변수 바인딩 `{context}` |
| 6 | LCEL 체인 | `prompt │ llm │ parser`, `.stream()` |
| 7 | RAG 체인 | `RunnableLambda`, 검색→생성 자동화 |
| 8 | 에러 처리 | 입력 검증, `try/except`, `safe_ask` |
| 9 | Gradio UI | `gr.ChatInterface`, `share=True` |
| 10 | 최종 통합 | `full_pipeline`, 성능 측정, 배포 |

**다음 주: 벡터 임베딩 + Naive RAG 파이프라인**
- NLP 기초: Tokenization, Embedding (BoW, TF-IDF, Word2Vec)
- LangChain RAG 컴포넌트: 문서 로더, 텍스트 분할, Vector Stores
- 키워드 검색(`search_faq`) → 벡터 유사도 검색으로 업그레이드